# Resiliency Intelligence — EDA & Model Evaluation

**Capital One — Resiliency Intelligence Team**

This notebook walks through:
1. Exploratory Data Analysis (EDA) of synthetic credit hardship customers
2. Feature engineering & preprocessing
3. XGBoost default risk classifier — training & evaluation (ROC, confusion matrix, feature importance)
4. Q-Learning RL agent — training & offer policy analysis
5. Business metrics summary

In [ ]:
import sys
sys.path.insert(0, '..')  # allow importing resiliency from the project root

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# resiliency library
from resiliency.data.generator import CustomerDataGenerator, GeneratorConfig, LABEL_COL, HARDSHIP_SEVERITY_COL
from resiliency.utils.preprocessing import FeaturePreprocessor
from resiliency.models.classifier import DefaultRiskClassifier
from resiliency.models.rl_agent import QLearningAgent, OfferType, OFFER_LABELS, discretise_state
from resiliency.evaluation.metrics import (
    plot_roc_curve, plot_confusion_matrix, classification_report_df,
    plot_feature_importance, business_metrics,
)

pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', '{:.3f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline
print('Libraries loaded ✓')

---
## 1. Generate Synthetic Dataset

In [ ]:
gen = CustomerDataGenerator(GeneratorConfig(n_samples=10_000, random_seed=42))
df = gen.generate()
train_df, test_df = gen.train_test_split(df, test_size=0.20)

print(f'Full dataset : {df.shape}')
print(f'Train        : {train_df.shape}  | default_rate={train_df[LABEL_COL].mean():.1%}')
print(f'Test         : {test_df.shape}   | default_rate={test_df[LABEL_COL].mean():.1%}')
df.head(3)

---
## 2. Exploratory Data Analysis

In [ ]:
# ---- 2.1 Summary statistics
df.describe().T.style.background_gradient(cmap='Blues', subset=['mean', 'std'])

In [ ]:
# ---- 2.2 Default rate by hardship severity
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Default rate by severity
sev_rate = df.groupby(HARDSHIP_SEVERITY_COL)[LABEL_COL].mean()
sev_rate.plot.bar(ax=axes[0], color=['#2ecc71', '#f39c12', '#e74c3c'], edgecolor='white')
axes[0].set_title('Default Rate by Hardship Severity', fontweight='bold')
axes[0].set_xlabel('Hardship Severity (0=Low, 1=Med, 2=High)')
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[0].set_xticklabels(['Low', 'Medium', 'High'], rotation=0)

# Credit score distribution by default status
for label, grp in df.groupby(LABEL_COL):
    axes[1].hist(grp['credit_score'], bins=40, alpha=0.6, 
                 label=f'Default={label}', density=True)
axes[1].set_title('Credit Score Distribution by Default', fontweight='bold')
axes[1].set_xlabel('Credit Score')
axes[1].legend()

# Debt-to-income by default
df_plot = df[df['debt_to_income_ratio'] < 3]
for label, grp in df_plot.groupby(LABEL_COL):
    axes[2].hist(grp['debt_to_income_ratio'], bins=40, alpha=0.6,
                 label=f'Default={label}', density=True)
axes[2].set_title('Debt-to-Income by Default', fontweight='bold')
axes[2].set_xlabel('Debt-to-Income Ratio')
axes[2].legend()

fig.tight_layout()
plt.show()

In [ ]:
# ---- 2.3 Correlation heatmap (top risk features)
risk_features = [
    'credit_score', 'credit_utilization_pct', 'months_delinquent',
    'debt_to_income_ratio', 'num_missed_payments_12m',
    'consecutive_missed_payments', 'has_bankruptcy', 'min_payment_ratio',
    LABEL_COL
]
corr = df[risk_features].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
    annot_kws={'size': 9}
)
ax.set_title('Correlation Matrix — Risk Features', fontsize=14, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# ---- 2.4 Employment status breakdown
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

emp_labels = {0: 'Unemployed', 1: 'Part-time', 2: 'Full-time', 3: 'Self-employed'}
df['emp_label'] = df['employment_status'].map(emp_labels)

emp_counts = df['emp_label'].value_counts()
emp_counts.plot.pie(ax=axes[0], autopct='%1.1f%%', startangle=90,
                    colors=['#e74c3c', '#f39c12', '#2ecc71', '#3498db'])
axes[0].set_title('Employment Status Distribution', fontweight='bold')
axes[0].set_ylabel('')

emp_default = df.groupby('emp_label')[LABEL_COL].mean().sort_values(ascending=False)
emp_default.plot.barh(ax=axes[1], color='#004977', edgecolor='white')
axes[1].set_title('Default Rate by Employment Status', fontweight='bold')
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
axes[1].set_xlabel('Default Rate')

fig.tight_layout()
plt.show()

In [ ]:
# ---- 2.5 Delinquency vs default (box plot)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

df.boxplot(column='months_delinquent', by=LABEL_COL, ax=axes[0],
           notch=True, patch_artist=True,
           boxprops=dict(facecolor='#004977', alpha=0.7))
axes[0].set_title('Months Delinquent by Default Status', fontweight='bold')
axes[0].set_xlabel('Default (0=No, 1=Yes)')
axes[0].set_ylabel('Months Delinquent')
plt.suptitle('')  # remove pandas auto-title

df.boxplot(column='credit_utilization_pct', by=LABEL_COL, ax=axes[1],
           notch=True, patch_artist=True,
           boxprops=dict(facecolor='#D03027', alpha=0.7))
axes[1].set_title('Credit Utilization by Default Status', fontweight='bold')
axes[1].set_xlabel('Default (0=No, 1=Yes)')
axes[1].set_ylabel('Utilization %')
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.suptitle('')

fig.tight_layout()
plt.show()

---
## 3. XGBoost Default Risk Classifier

In [ ]:
# ---- 3.1 Prepare features
X_train = train_df.drop(columns=[LABEL_COL, HARDSHIP_SEVERITY_COL])
y_train = train_df[LABEL_COL]
X_test  = test_df.drop(columns=[LABEL_COL, HARDSHIP_SEVERITY_COL])
y_test  = test_df[LABEL_COL]

print(f'X_train: {X_train.shape}  |  Class balance (train): {y_train.value_counts().to_dict()}')

In [ ]:
# ---- 3.2 Train classifier
clf = DefaultRiskClassifier(calibrate=True, classification_threshold=0.45)
clf.fit(X_train, y_train, eval_set=(X_test, y_test), verbose=False)
print(repr(clf))

In [ ]:
# ---- 3.3 ROC Curve
y_prob = clf.predict_proba(X_test)
y_pred = clf.predict(X_test)

fig = plot_roc_curve(y_test.values, y_prob, model_name='XGBoost (calibrated)')
plt.show()

In [ ]:
# ---- 3.4 Confusion Matrix
fig = plot_confusion_matrix(y_test.values, y_pred, normalize=True)
plt.show()

print('\nRaw counts:')
fig2 = plot_confusion_matrix(y_test.values, y_pred, normalize=False)
plt.show()

In [ ]:
# ---- 3.5 Classification Report
report = classification_report_df(y_test.values, y_pred)
report.style.background_gradient(cmap='Blues', subset=['precision', 'recall', 'f1-score'])

In [ ]:
# ---- 3.6 Feature Importance
fig = plot_feature_importance(clf.feature_names_, clf.feature_importances_, top_n=20)
plt.show()

In [ ]:
# ---- 3.7 Probability distribution
fig, ax = plt.subplots(figsize=(10, 5))
for label in [0, 1]:
    mask = y_test.values == label
    ax.hist(y_prob[mask], bins=50, alpha=0.6, density=True,
            label=f'Actual default={label}')
ax.axvline(clf.classification_threshold, color='red', linestyle='--',
           label=f'Threshold = {clf.classification_threshold}')
ax.set_xlabel('Predicted Default Probability', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Predicted Probability Distribution by True Label', fontweight='bold')
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# ---- 3.8 Business Metrics
biz = business_metrics(y_test.values, y_prob, y_pred)
biz.T.rename(columns={0: 'Value'}).style.set_caption('Business Impact Summary')

---
## 4. Threshold Sensitivity Analysis

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = np.arange(0.10, 0.90, 0.02)
records = []
for t in thresholds:
    preds = (y_prob >= t).astype(int)
    records.append({
        'threshold': t,
        'precision': precision_score(y_test, preds, zero_division=0),
        'recall':    recall_score(y_test, preds, zero_division=0),
        'f1':        f1_score(y_test, preds, zero_division=0),
    })
thresh_df = pd.DataFrame(records)

fig, ax = plt.subplots(figsize=(10, 5))
for col, color in [('precision', '#004977'), ('recall', '#D03027'), ('f1', '#F5A623')]:
    ax.plot(thresh_df['threshold'], thresh_df[col], lw=2, label=col.title(), color=color)
ax.axvline(0.45, color='gray', linestyle='--', label='Chosen threshold (0.45)')
ax.set_xlabel('Classification Threshold')
ax.set_ylabel('Score')
ax.set_title('Precision / Recall / F1 vs. Threshold', fontweight='bold')
ax.legend()
fig.tight_layout()
plt.show()

---
## 5. Reinforcement Learning — Q-Learning Agent

In [ ]:
# ---- 5.1 Compute default probabilities for all customers
all_X = df.drop(columns=[LABEL_COL, HARDSHIP_SEVERITY_COL])
all_probs = clf.predict_proba(all_X)
print(f'Default probability range: {all_probs.min():.3f} – {all_probs.max():.3f}')

In [ ]:
# ---- Reload resiliency modules to pick up any source changes without restarting kernel
import importlib
import resiliency.models.rl_agent as _rl_mod

importlib.reload(_rl_mod)

from resiliency.models.rl_agent import QLearningAgent, OfferType, OFFER_LABELS, discretise_state
print("resiliency.models.rl_agent reloaded ✓")

In [ ]:
# ---- 5.2 Train Q-learning agent
agent = QLearningAgent(
    alpha=0.10,
    gamma=0.95,
    epsilon_start=1.0,
    epsilon_min=0.05,
    epsilon_decay=0.9995,
)
agent.train(df, default_probs=all_probs, n_episodes=15_000, log_every=3_000)

In [ ]:
# ---- 5.3 Training reward curve
fig = agent.plot_reward_history(window=300)
plt.show()

In [ ]:
# ---- 5.4 Policy analysis — recommended action by risk tier
recommendations = []
for idx, row in test_df.iterrows():
    p = float(all_probs[df.index.get_loc(idx)] if idx in df.index else 0.5)

# Easier: score test_df then analyse
test_X = test_df.drop(columns=[LABEL_COL, HARDSHIP_SEVERITY_COL])
test_probs = clf.predict_proba(test_X)
risk_tiers = np.where(test_probs < 0.30, 'LOW', np.where(test_probs < 0.60, 'MEDIUM', 'HIGH'))

recs = []
for i, row in test_df.iterrows():
    p = float(test_probs[test_df.index.get_loc(i)])
    rec = agent.recommend(row.to_dict(), default_prob=p)
    recs.append({'offer_type': rec['offer_type'], 'default_prob': p,
                 'risk_tier': 'LOW' if p < 0.3 else ('MEDIUM' if p < 0.6 else 'HIGH')})

rec_df = pd.DataFrame(recs)

offer_by_tier = rec_df.groupby(['risk_tier', 'offer_type']).size().unstack(fill_value=0)
offer_by_tier_pct = offer_by_tier.div(offer_by_tier.sum(axis=1), axis=0)

fig, ax = plt.subplots(figsize=(12, 5))
offer_by_tier_pct.loc[['LOW', 'MEDIUM', 'HIGH']].plot.bar(
    ax=ax, stacked=True,
    colormap='tab10',
    edgecolor='white',
    width=0.6,
)
ax.set_title('Recommended Offer Distribution by Risk Tier', fontsize=14, fontweight='bold')
ax.set_xlabel('Risk Tier', fontsize=12)
ax.set_ylabel('Proportion of Recommendations', fontsize=12)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(title='Offer Type', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xticklabels(['LOW', 'MEDIUM', 'HIGH'], rotation=0)
fig.tight_layout()
plt.show()

In [ ]:
# ---- 5.5 Sample recommendations
print('=== Sample Debt Resolution Recommendations ===')
for scenario_name, filter_fn in [
    ('HIGH RISK — Unemployed with bankruptcy',
     lambda r: r['employment_status'] == 0 and r['has_bankruptcy'] == 1),
    ('MEDIUM RISK — Requested hardship program',
     lambda r: r['requested_hardship_program'] == 1 and r['months_delinquent'] < 4),
    ('LOW RISK — Good credit, minor delinquency',
     lambda r: r['credit_score'] > 650 and r['months_delinquent'] <= 1),
]:
    subset = test_df[test_df.apply(filter_fn, axis=1)]
    if len(subset) == 0:
        print(f'\n[{scenario_name}] — No matching customers')
        continue
    sample_row = subset.iloc[0]
    sample_idx = subset.index[0]
    p = float(test_probs[test_df.index.get_loc(sample_idx)])
    rec = agent.recommend(sample_row.to_dict(), default_prob=p)
    print(f'\n[{scenario_name}]')
    print(f'  Default probability : {p:.1%}')
    print(f'  Recommended offer   : {rec["offer_label"]}')
    print(f'  Confidence          : {rec["confidence"]:.1%}')
    print(f'  Q-values            : {rec["q_values"]}')

---
## 6. End-to-End Pipeline Summary

| Component | Details |
|-----------|--------|
| **Dataset** | 10,000 synthetic customers; ~22% default rate |
| **Classifier** | XGBoost + Platt calibration; calibrated probabilities |
| **RL Agent** | Tabular Q-learning; 6 actions; 15,000 training episodes |
| **API** | FastAPI; `/score`, `/recommend`, `/score-and-recommend` |
| **Top Risk Drivers** | Months delinquent, DTI, credit score, missed payments |
| **Offer Policy** | High-risk → Settlement / Payment Plan; Low-risk → Skip Payment |


In [ ]:
# ---- Final: save models from notebook
import os
os.makedirs('../models', exist_ok=True)
clf.save('../models/default_risk_classifier.pkl')
agent.save('../models/rl_agent.pkl')
print('Models saved to ../models/')